### *Matérn Correlation Matrix generation* 

**Goal**: is to generate the common correlation structure $R$ jointly from three precipitation data sets.To that end, we fix the Matérn  parameters in the correlations function (length scale "$l_s$", smoothness "$k$").

In [1]:
import numpy as np
import pandas as pd
import os
from scipy.spatial.distance import pdist, squareform
from scipy.special import kv, gamma
import glob
import rioxarray
import warnings
warnings.filterwarnings('ignore')

- define helper functions

In [2]:
def get_coords_vals(raster_path):
    "func to extract coordinates and values from raster data"
    with rioxarray.open_rasterio(raster_path) as rds:
        data = rds.squeeze().values
        x_coords, y_coords = np.meshgrid(rds.x.values, rds.y.values)
        coords = np.column_stack((x_coords.ravel(), y_coords.ravel()))
        mask = ~np.isnan(data.ravel())
        return coords[mask], data.ravel()[mask]

In [3]:
def matern_correlation(distance, phi, kappa):
    scaled_distance = np.maximum(distance / max(phi,1e-8), 1e-8)  # avoid division by 0
    cor = 1 / (2**(kappa-1) * gamma(kappa)) * (scaled_distance**kappa) * kv(kappa, scaled_distance)
    return np.maximum(cor, 0)  # +ve

In [4]:
def create_R_matern(phi, kappa, distances, R=None):
    RR = matern_correlation(distances, phi, kappa)
    if R is None:
        R = squareform(RR)
    else:
        R[:] = squareform(RR)
    np.fill_diagonal(R, 1.0)
    return R

- generate the common correlation matrix using fixed parameters

In [5]:
ls_fixed = 50  # km
k_fixed = 1.5

    
template = '../data/p/trmm/trmm_2003.01.01.tif'   
coords, _ = get_coords_vals(template)
distances = pdist(coords, metric='euclidean') * 111  # km

# R
R = create_R_matern(ls_fixed, k_fixed, distances)
#save
#pd.DataFrame(R).to_csv('../output/R_matrix.csv', index=False)